# 10 · Graph and knowledge exports (Mexico)
**Country series · Mexico** · *generated from `notebooks/country_template`: edit the template*

The property graph, temporal events, federated silos and GraphRAG documents of Mexico (pipeline series,
notebook 10). The bank-wide replay found that tokenising a missing IP gives one constant token, a supernode. Here:
how large that supernode is inside one country, and what genuine sharing remains.

In [1]:
import sys
import time

sys.path.insert(0, "../../src")
import plotly.express as px
from IPython.display import Markdown, display
from itables import show

from latam_eda import country, theme

COUNTRY = "MX"
DATASET = "main"
PREFIX = "country"
CTRY = country.SCOPES[COUNTRY]
theme.register()
t0 = time.time()
pl = country.session(COUNTRY, DATASET)
country.prepare(pl, "gold")
GRAPH = [
    "graph_nodes",
    "graph_edges",
    "tgn_node_features",
    "tgn_tx_events",
    "fgl_silo_nodes",
    "fgl_silo_edges",
    "kb_entity_docs",
    "kb_entity_triples",
]
built = pl.build_set(GRAPH)

  external  fgl_silo_nodes                                        74,931 rows     0.4s
  external  fgl_silo_edges                                       345,597 rows     0.1s
  external  graph_nodes                                          309,598 rows     1.4s
  external  graph_edges                                        2,313,293 rows     3.5s
  external  kb_entity_docs                                       108,767 rows     0.9s
  external  kb_entity_triples                                  2,230,038 rows     0.8s
  external  tgn_node_features                                     74,931 rows     0.1s
  external  tgn_tx_events                                        499,798 rows     0.5s


In [2]:
show(
    pl.q("select node_type, count(*) as nodes from {graph_nodes} group by 1 order by 2 desc"),
    paging=False,
)
deg = pl.q(
    """select dst_id, count(distinct src_id) as customers from {graph_edges} where dst_type = 'ip' group by 1"""
)
custs = pl.q("select count(*) from {stg_customers}").iloc[0, 0]
top = int(deg.customers.max()) if len(deg) else 0
custs = max(int(custs), 1)
fig = px.histogram(
    deg, x="customers", log_y=True, nbins=40, title=f"{CTRY.title}: customers per IP node"
)
fig.update_layout(height=280)
fig.show()
display(
    Markdown(
        f"**The largest IP node links {top:,} of {CTRY.name}'s {custs:,} customers ({100 * top / custs:.0f} %)**; "
        f"{int((deg.customers == 2).sum())} IP nodes link exactly two customers. The large node is the token of NULL "
        "(missing IPs), the same defect as bank-wide: fix `pii_hash` before any graph work in any country."
    )
)

**The largest IP node links 71,469 of Mexico's 74,907 customers (95 %)**; 52 IP nodes link exactly two customers. The large node is the token of NULL (missing IPs), the same defect as bank-wide: fix `pii_hash` before any graph work in any country.

In [3]:
show(
    pl.q("""select silo, residency_region, count(*) as nodes from {fgl_silo_nodes} group by all"""),
    paging=False,
)
show(
    pl.q(
        """select entity_type, count(*) as documents from {kb_entity_docs} group by 1 order by 2 desc"""
    ),
    paging=False,
)

A country lake produces exactly one federated silo, in the country's residency region: the federated design and the
country cut agree, which is the point of putting residency in the data model.

## Findings for Mexico and what to do
The computed statements above; the token fix and the missing transfer counterparties are bank-wide blockers for
graph learning in every country.

In [4]:
pl.close()
print(f"notebook time {time.time() - t0:.0f}s")

notebook time 9s
